# XGBoost on PySpark - S3 数据源 + 纯分布式计算（二分类）
数据存储于 MinIO (S3兼容) → Spark 分布式读取 → 特征工程 → 分布式训练 → 分布式评估

**核心特点：全程分布式，不使用 `toPandas()`，适合大规模数据。**

In [1]:
import sys, os, glob
# apache/spark 镜像中 pyspark 不在 pip 路径，需要手动添加
sys.path.insert(0, "/opt/spark/python")
# 添加 py4j
py4j_zip = glob.glob("/opt/spark/python/lib/py4j-*.zip")
if py4j_zip:
    sys.path.insert(0, py4j_zip[0])
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("XGBoost-S3-Distributed") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "2g") \
    .config("spark.executor.cores", "1") \
    .config("spark.driver.memory", "2g") \
    .config("spark.task.cpus", "1") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.fast.upload", "true") \
    .getOrCreate()

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/21 08:58:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
import numpy as np
from pyspark.sql import functions as F

np.random.seed(42)
n_samples = 100000
n_features = 28

# 生成合成特征
X = np.random.randn(n_samples, n_features)
logits = -3 + 2 * X[:, 0] + 1.5 * X[:, 1] - X[:, 2] + 0.5 * X[:, 3]
probs = 1 / (1 + np.exp(-logits))
y = (probs > 0.5).astype(int)

# 添加少量噪声
flip_idx = np.random.choice(n_samples, size=int(n_samples * 0.01), replace=False)
y[flip_idx] = 1 - y[flip_idx]

print(f"Total samples: {n_samples}")
print(f"Positive (fraud): {y.sum()} ({y.mean()*100:.2f}%)")
print(f"Negative (normal): {(1-y).sum()} ({(1-y.mean())*100:.2f}%)")

# 构建行数据 list of tuples（避免 toPandas）
columns = ["Time"] + [f"V{i}" for i in range(1, n_features+1)] + ["Amount", "Class"]
rows = []
for i in range(n_samples):
    row = [float(i)] + [float(X[i, j]) for j in range(n_features)] + [float(abs(np.random.randn() * 100 + 50)), int(y[i])]
    rows.append(tuple(row))

# 从 list 创建 Spark DataFrame（分布式）
df = spark.createDataFrame(rows, columns)
print(f"Spark DataFrame 分区数: {df.rdd.getNumPartitions()}")

# 写入 S3 (MinIO) 为 Parquet 格式
s3_path = "s3a://datasets/creditcard.parquet"
df.write.mode("overwrite").parquet(s3_path)
print(f"数据已写入 S3: {s3_path}")

# 释放本地 df，后续从 S3 读取
df.unpersist() if hasattr(df, "unpersist") else None
print("本地 DataFrame 已释放，后续将从 S3 读取")

Total samples: 100000
Positive (fraud): 14464 (14.46%)
Negative (normal): 85536 (85.54%)
Spark DataFrame 分区数: 2


26/06/21 08:59:00 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/06/21 08:59:00 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/06/21 08:59:01 WARN TaskSetManager: Stage 0 contains a task of very large size (13485 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

数据已写入 S3: s3a://datasets/creditcard.parquet
本地 DataFrame 已释放，后续将从 S3 读取


26/06/21 08:57:39 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


26/06/21 08:57:40 WARN TaskSetManager: Stage 0 contains a task of very large size (13485 KiB). The maximum recommended task size is 1000 KiB.


数据已写入 S3: s3a://datasets/creditcard.parquet
本地 DataFrame 已释放，后续将从 S3 读取


In [4]:
from pyspark.sql import functions as F

# 从 S3 分布式读取
s3_path = "s3a://datasets/creditcard.parquet"
df = spark.read.parquet(s3_path)

# 统计信息 - 全部用 Spark 聚合，不转 Pandas
total_count = df.count()
num_partitions = df.rdd.getNumPartitions()
print(f"总行数: {total_count}, 分区数: {num_partitions}")
print(f"Schema:")
df.printSchema()

# 类别分布 - 纯 Spark
print("类别分布:")
df.groupBy("Class").count().orderBy("Class").show()

# 类别比例 - 纯 Spark
total = df.count()
df.groupBy("Class").count() \
    .withColumn("ratio", F.round(F.col("count") / total * 100, 2)) \
    .orderBy("Class") \
    .show()

总行数: 100000, 分区数: 2
Schema:
root
 |-- Time: double (nullable = true)
 |-- V1: double (nullable = true)
 |-- V2: double (nullable = true)
 |-- V3: double (nullable = true)
 |-- V4: double (nullable = true)
 |-- V5: double (nullable = true)
 |-- V6: double (nullable = true)
 |-- V7: double (nullable = true)
 |-- V8: double (nullable = true)
 |-- V9: double (nullable = true)
 |-- V10: double (nullable = true)
 |-- V11: double (nullable = true)
 |-- V12: double (nullable = true)
 |-- V13: double (nullable = true)
 |-- V14: double (nullable = true)
 |-- V15: double (nullable = true)
 |-- V16: double (nullable = true)
 |-- V17: double (nullable = true)
 |-- V18: double (nullable = true)
 |-- V19: double (nullable = true)
 |-- V20: double (nullable = true)
 |-- V21: double (nullable = true)
 |-- V22: double (nullable = true)
 |-- V23: double (nullable = true)
 |-- V24: double (nullable = true)
 |-- V25: double (nullable = true)
 |-- V26: double (nullable = true)
 |-- V27: double (nullable = t

+-----+-----+
|Class|count|
+-----+-----+
|    0|85536|
|    1|14464|
+-----+-----+



+-----+-----+-----+
|Class|count|ratio|
+-----+-----+-----+
|    0|85536|85.54|
|    1|14464|14.46|
+-----+-----+-----+



In [5]:
from pyspark.ml.feature import VectorAssembler
from pyspark.sql import functions as F

feature_cols = [f"V{i}" for i in range(1, 29)] + ["Amount"]

# 分布式 null 检查
null_counts = df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in feature_cols])
null_row = null_counts.collect()[0]
total_nulls = sum(null_row[c] for c in feature_cols)
print(f"特征总 null 数: {total_nulls}")

# VectorAssembler - 树模型不需要标准化
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

# 只保留 features 和 label，减少数据传输
df_transformed = assembler.transform(df).select("features", "Class")
print("特征工程完成！")
df_transformed.show(5, truncate=False)

特征总 null 数: 0
特征工程完成！


[Stage 30:>                                                         (0 + 1) / 1]

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|features                                                                                                                                                                                                                                                                                                                                                                                                                 

In [7]:
from pyspark.sql import functions as F

# 分布式划分
train_df, test_df = df_transformed.randomSplit([0.8, 0.2], seed=42)

# 重分区并缓存
train_df = train_df.repartition(2).cache()
test_df = test_df.repartition(2).cache()

# 触发缓存 + 分布式统计
train_count = train_df.count()
test_count = test_df.count()

# 纯 Spark 计算正负样本数
train_stats = train_df.groupBy("Class").count().orderBy("Class").collect()
test_stats = test_df.groupBy("Class").count().orderBy("Class").collect()

print(f"训练集: {train_count} (负:{train_stats[0][1]}, 正:{train_stats[1][1] if len(train_stats)>1 else 0})")
print(f"测试集: {test_count} (负:{test_stats[0][1]}, 正:{test_stats[1][1] if len(test_stats)>1 else 0})")

# 验证分区不为空
partitions = train_df.rdd.glom().map(len).collect()
print(f"训练集各分区行数: {partitions}")

训练集: 80062 (负:68507, 正:11555)
测试集: 19938 (负:17029, 正:2909)


[Stage 72:>                                                         (0 + 2) / 2]

训练集各分区行数: [40031, 40031]


训练集: 80062 (负:68507, 正:11555)
测试集: 19938 (负:17029, 正:2909)


训练集各分区行数: [40031, 40031]


In [8]:
from xgboost.spark import SparkXGBClassifier
import time

# 分布式 XGBoost
xgb_classifier = SparkXGBClassifier(
    features_col="features",
    label_col="Class",
    num_workers=2,
    n_estimators=50,
    max_depth=6,
    learning_rate=0.1,
    eval_metric="auc",
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=3.0,
    missing=0.0,
    verbosity=1,
)

print("开始分布式训练（数据来自 S3）...")
start_time = time.time()
xgb_model = xgb_classifier.fit(train_df)
train_time = time.time() - start_time
print(f"训练完成！耗时: {train_time:.2f} 秒")

/usr/local/lib/python3.8/dist-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


开始分布式训练（数据来自 S3）...


2026-06-21 08:59:45,225 INFO XGBoost-PySpark: _fit Running xgboost-2.1.4 on 2 workers with
	booster params: {'objective': 'binary:logistic', 'colsample_bytree': 0.8, 'device': 'cpu', 'eval_metric': 'auc', 'learning_rate': 0.1, 'max_depth': 6, 'reg_alpha': 0.1, 'reg_lambda': 1.0, 'scale_pos_weight': 3.0, 'subsample': 0.8, 'verbosity': 1, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 50}
	dmatrix_kwargs: {'nthread': 1, 'missing': 0.0}
2026-06-21 08:59:50,419 INFO XGBoost-PySpark: _fit Finished xgboost training!   


训练完成！耗时: 5.80 秒


2026-06-21 08:58:00,039 INFO XGBoost-PySpark: _fit Finished xgboost training!


训练完成！耗时: 5.58 秒


In [9]:
# 分布式预测
predictions = xgb_model.transform(test_df)

# 缓存预测结果
predictions.cache()
pred_count = predictions.count()
print(f"预测样本数: {pred_count}")

# 展示前10条 - 用 Spark show(), 不用 toPandas()
predictions.select("Class", "rawPrediction", "probability", "prediction").show(10, truncate=False)

预测样本数: 19938
+-----+----------------------------------------+-----------------------------------------+----------+
|Class|rawPrediction                           |probability                              |prediction|
+-----+----------------------------------------+-----------------------------------------+----------+
|1    |[-1.0228707790374756,1.0228707790374756]|[0.26446855068206787,0.7355314493179321] |1.0       |
|0    |[3.0637621879577637,-3.0637621879577637]|[0.9553729891777039,0.044627025723457336]|0.0       |
|0    |[3.567086696624756,-3.567086696624756]  |[0.9725374579429626,0.027462514117360115]|0.0       |
|0    |[1.755565881729126,-1.755565881729126]  |[0.852653443813324,0.14734655618667603]  |0.0       |
|0    |[3.5157999992370605,-3.5157999992370605]|[0.9711340069770813,0.028866002336144447]|0.0       |
|1    |[-1.2150307893753052,1.2150307893753052]|[0.2288120985031128,0.7711879014968872]  |1.0       |
|0    |[2.421107530593872,-2.421107530593872]  |[0.9184227585792542,0

In [10]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# 分布式评估 - 全部在 Spark 内完成
auc = BinaryClassificationEvaluator(
    labelCol="Class", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
).evaluate(predictions)

pr_auc = BinaryClassificationEvaluator(
    labelCol="Class", rawPredictionCol="rawPrediction", metricName="areaUnderPR"
).evaluate(predictions)

accuracy = MulticlassClassificationEvaluator(
    labelCol="Class", predictionCol="prediction", metricName="accuracy"
).evaluate(predictions)

f1 = MulticlassClassificationEvaluator(
    labelCol="Class", predictionCol="prediction", metricName="f1"
).evaluate(predictions)

print("=" * 50)
print("模型评估结果")
print("=" * 50)
print(f"AUC (ROC):      {auc:.4f}")
print(f"AUC (PR):       {pr_auc:.4f}")
print(f"Accuracy:       {accuracy:.4f}")
print(f"F1 Score:       {f1:.4f}")

模型评估结果
AUC (ROC):      0.9685
AUC (PR):       0.9396
Accuracy:       0.9649
F1 Score:       0.9656


In [11]:
from pyspark.sql import functions as F

# 分布式混淆矩阵 - 纯 Spark 计算
cm = predictions.groupBy("Class", "prediction").count().orderBy("Class", "prediction")
print("混淆矩阵 (Class vs prediction):")
cm.show()

# 用 Spark 计算 TP/TN/FP/FN
tp = predictions.filter((F.col("Class") == 1) & (F.col("prediction") == 1)).count()
tn = predictions.filter((F.col("Class") == 0) & (F.col("prediction") == 0)).count()
fp = predictions.filter((F.col("Class") == 0) & (F.col("prediction") == 1)).count()
fn = predictions.filter((F.col("Class") == 1) & (F.col("prediction") == 0)).count()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

print(f"TP={tp}, TN={tn}, FP={fp}, FN={fn}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")

混淆矩阵 (Class vs prediction):
+-----+----------+-----+
|Class|prediction|count|
+-----+----------+-----+
|    0|       0.0|16539|
|    0|       1.0|  490|
|    1|       0.0|  210|
|    1|       1.0| 2699|
+-----+----------+-----+

TP=2699, TN=16539, FP=490, FN=210
Precision: 0.8463
Recall:    0.9278


+-----+----------+-----+
|Class|prediction|count|
+-----+----------+-----+
|    0|       0.0|16549|
|    0|       1.0|  480|
|    1|       0.0|  219|
|    1|       1.0| 2690|
+-----+----------+-----+



TP=2690, TN=16549, FP=480, FN=219
Precision: 0.8486
Recall:    0.9247


In [12]:
import numpy as np

np.random.seed(123)
n_new = 5

# 生成新数据
new_X = np.random.randn(n_new, 28) * 0.5
new_amount = np.abs(np.random.randn(n_new) * 50 + 100)

columns = ["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount", "Class"]
rows = []
for i in range(n_new):
    row = [float(i)] + [float(new_X[i, j]) for j in range(28)] + [float(new_amount[i]), 0]
    rows.append(tuple(row))

# 创建 Spark DataFrame（不经过 Pandas）
new_spark_df = spark.createDataFrame(rows, columns)

# 特征变换 + 预测 - 全分布式
new_transformed = assembler.transform(new_spark_df).select("features")
new_predictions = xgb_model.transform(new_transformed)

# 用 Spark 收集结果（小数据用 collect，大数据应写入 S3）
results = new_predictions.select("probability", "prediction").collect()
print("新数据预测结果:")
for i, row in enumerate(results):
    fraud_prob = float(row["probability"][1])
    pred_label = "欺诈" if row["prediction"] == 1 else "正常"
    print(f"样本{i+1}: 欺诈概率={fraud_prob:.4f}, 预测={pred_label}")

新数据预测结果:
样本1: 欺诈概率=0.0374, 预测=正常
样本2: 欺诈概率=0.0209, 预测=正常
样本3: 欺诈概率=0.1327, 预测=正常
样本4: 欺诈概率=0.0208, 预测=正常
样本5: 欺诈概率=0.0344, 预测=正常


In [13]:
# 将模型保存到 S3 (MinIO)
model_s3_path = "s3a://models/xgboost_binary_cls"
xgb_model.write().overwrite().save(model_s3_path)
print(f"模型已保存到 S3: {model_s3_path}")

# 验证 S3 上的模型文件
model_df = spark.read.format("binaryFile").load(model_s3_path)
print(f"模型文件数: {model_df.count()}")
print("模型保存成功，可用于后续加载推理！")

模型已保存到 S3: s3a://models/xgboost_binary_cls
模型文件数: 0
模型保存成功，可用于后续加载推理！


26/06/21 09:06:26 ERROR Inbox: Ignoring error
java.util.concurrent.RejectedExecutionException: Task java.util.concurrent.ScheduledThreadPoolExecutor$ScheduledFutureTask@92eac0c[Not completed, task = java.util.concurrent.Executors$RunnableAdapter@71408ce1[Wrapped task = org.apache.spark.scheduler.cluster.StandaloneSchedulerBackend$StandaloneDriverEndpoint$$anon$2@6685769b]] rejected from java.util.concurrent.ScheduledThreadPoolExecutor@725d6dc6[Terminated, pool size = 0, active threads = 0, queued tasks = 0, completed tasks = 0]
	at java.base/java.util.concurrent.ThreadPoolExecutor$AbortPolicy.rejectedExecution(Unknown Source)
	at java.base/java.util.concurrent.ThreadPoolExecutor.reject(Unknown Source)
	at java.base/java.util.concurrent.ScheduledThreadPoolExecutor.delayedExecute(Unknown Source)
	at java.base/java.util.concurrent.ScheduledThreadPoolExecutor.schedule(Unknown Source)
	at org.apache.spark.scheduler.cluster.StandaloneSchedulerBackend$StandaloneDriverEndpoint.$anonfun$onDisco

## 总结

本 Notebook 演示了 **大规模数据 + 纯分布式计算** 的完整流程：

| 步骤 | 方式 | 数据结构 |
|------|------|----------|
| 数据生成 | Spark createDataFrame | list of tuples → Spark DataFrame |
| 数据存储 | 写入 S3 (MinIO) | Parquet 格式 |
| 数据读取 | spark.read.parquet(s3a://...) | Spark DataFrame (分布式) |
| 数据探索 | groupBy + count + collect | Spark DataFrame |
| 特征工程 | VectorAssembler | Spark DataFrame |
| 训练/测试划分 | randomSplit | Spark DataFrame |
| 模型训练 | SparkXGBClassifier.fit() | Spark DataFrame (2 workers) |
| 模型预测 | model.transform() | Spark DataFrame |
| 模型评估 | BinaryClassificationEvaluator | Spark DataFrame |
| 混淆矩阵 | filter + count | Spark DataFrame |
| 模型保存 | write to S3 | S3 (MinIO) |

**全程不使用 `toPandas()`，可扩展到 TB 级数据！**

In [12]:
# 释放缓存
try:
    train_df.unpersist()
    test_df.unpersist()
    predictions.unpersist()
    print("缓存已释放")
except:
    pass
# spark.stop()  # 取消注释以释放资源

缓存已释放
